# Titanic Dataset — Exploratory Data Analysis (EDA)

**Objective:** Explore the Titanic passenger dataset using descriptive statistics and visualizations, identify patterns, trends, and anomalies, and document findings.

**Libraries:** Pandas, NumPy, Matplotlib, Seaborn.

> Run the notebook from top to bottom. The dataset loader first tries Seaborn's Titanic dataset and then tries a public CSV fallback. An internet connection may be required the first time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

## 1. Load the dataset

In [ ]:
try:
    df = sns.load_dataset("titanic")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    raw = pd.read_csv(url)
    # Normalize common Kaggle column names to a consistent set where possible
    df = raw.rename(columns={
        "Survived": "survived", "Pclass": "pclass", "Sex": "sex",
        "Age": "age", "SibSp": "sibsp", "Parch": "parch",
        "Fare": "fare", "Embarked": "embarked", "Name": "name",
        "Ticket": "ticket", "Cabin": "cabin", "PassengerId": "passenger_id"
    })

df.head()

## 2. Understand the data: shape, columns, types

In [ ]:
print("Rows and columns:", df.shape)
display(df.head())
display(df.info())

**Observation guide:** Record the number of rows and columns, identify numeric versus categorical columns, and note columns with missing values. The exact column names depend on which dataset source loaded successfully.

In [ ]:
# Descriptive statistics for numeric columns
display(df.describe(include="all").T)

# Missing-value counts and percentages
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)
display(missing)

## 3. Univariate analysis: categorical columns

In [ ]:
# Choose whichever categorical columns are available in this version of the dataset
for col in ["survived", "sex", "pclass", "embarked", "class", "who", "alone"]:
    if col in df.columns:
        print(f"\nValue counts for {col}:")
        display(df[col].value_counts(dropna=False))

**Observation guide:** `value_counts()` shows the most common categories. In the standard Titanic dataset, more passengers died than survived, more passengers were male than female, and third class contained the largest number of passengers.

## 4. Survival rate by sex and passenger class

In [ ]:
# Survival rate by sex
if {"sex", "survived"}.issubset(df.columns):
    sex_rate = df.groupby("sex", observed=False)["survived"].mean().sort_values(ascending=False)
    display((sex_rate * 100).round(1).rename("survival_rate_percent").to_frame())
    sns.barplot(data=df, x="sex", y="survived", errorbar=None)
    plt.title("Survival Rate by Sex")
    plt.ylabel("Survival rate")
    plt.xlabel("Sex")
    plt.ylim(0, 1)
    plt.show()

# Survival rate by passenger class
class_col = "pclass" if "pclass" in df.columns else ("class" if "class" in df.columns else None)
if class_col and "survived" in df.columns:
    class_rate = df.groupby(class_col, observed=False)["survived"].mean()
    display((class_rate * 100).round(1).rename("survival_rate_percent").to_frame())
    sns.barplot(data=df, x=class_col, y="survived", errorbar=None, order=sorted(df[class_col].dropna().unique(), key=str))
    plt.title("Survival Rate by Passenger Class")
    plt.ylabel("Survival rate")
    plt.xlabel("Passenger class")
    plt.ylim(0, 1)
    plt.show()

**Observation:** In the commonly used Titanic dataset, women had a substantially higher survival rate than men. Survival also tended to be higher in first class than in second or third class. These are associations in the data, not proof that sex or class alone caused survival.

## 5. Histograms: distributions of age and fare

In [ ]:
for col in ["age", "fare"]:
    if col in df.columns:
        sns.histplot(data=df, x=col, kde=True, bins=30)
        plt.title(f"Distribution of {col.title()}")
        plt.xlabel(col.title())
        plt.ylabel("Passenger count")
        plt.show()

**Observation guide:** Age values are concentrated among children and adults, while fare is typically right-skewed: most passengers paid lower fares and a smaller number paid much higher fares.

## 6. Boxplots: outlier exploration

In [ ]:
for col in ["age", "fare"]:
    if col in df.columns:
        sns.boxplot(data=df, x=col)
        plt.title(f"Boxplot of {col.title()}")
        plt.show()

**Observation:** Fare commonly contains high-value outliers. These can reflect differences in ticket class, cabin arrangements, or group tickets. An outlier is not automatically an error; investigate before removing it.

## 7. Scatterplot: age versus fare, colored by survival

In [ ]:
if {"age", "fare", "survived"}.issubset(df.columns):
    sns.scatterplot(data=df, x="age", y="fare", hue="survived", alpha=0.7)
    plt.title("Age vs Fare by Survival")
    plt.xlabel("Age")
    plt.ylabel("Fare")
    plt.show()

**Observation guide:** Look for overlap between survivors and non-survivors, clusters by fare, and any unusual points. A scatterplot may show patterns but cannot establish causation.

## 8. Pairplot: relationships among selected numeric variables

In [ ]:
pair_cols = [c for c in ["survived", "age", "fare", "sibsp", "parch", "pclass"] if c in df.columns]
if len(pair_cols) >= 2:
    sns.pairplot(df[pair_cols].dropna().sample(min(400, len(df.dropna(subset=pair_cols))), random_state=42),
                 corner=True, diag_kind="hist")
    plt.suptitle("Pairplot of Selected Numeric Variables", y=1.02)
    plt.show()

**Observation guide:** Pairplots help compare numeric distributions and pairwise relationships. Because the dataset mixes discrete codes and continuous measures, interpret plots involving `survived` and `pclass` as category-like comparisons rather than continuous measurements.

## 9. Correlation heatmap

In [ ]:
numeric = df.select_dtypes(include=np.number)
if not numeric.empty:
    corr = numeric.corr(numeric_only=True)
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
    plt.title("Correlation Heatmap of Numeric Features")
    plt.show()
    display(corr.round(2))

**Observation:** Correlation measures linear association between numeric columns. In the standard Titanic data, passenger class and fare are related, and survival has a positive association with fare and a negative association with passenger-class number (where 1 is first class and 3 is third class). Correlation does not imply causation.

## 10. Missing values and data quality

In [ ]:
# Visualize missingness
plt.figure(figsize=(10, 5))
sns.heatmap(df.isna(), cbar=False, yticklabels=False)
plt.title("Missing Values by Row and Column")
plt.xlabel("Columns")
plt.show()

print("Duplicate rows:", df.duplicated().sum())

**Observation:** Age and cabin/deck information are commonly incomplete. Missingness should be reported and handled based on the analysis goal. For example, age could be imputed with a median for a modeling task, but for EDA it is often better to report missingness and avoid silently altering the raw data.

## 11. Final summary

In [ ]:
# Compute key metrics from the loaded dataset
if "survived" in df.columns:
    print("Overall survival rate:", round(df["survived"].mean() * 100, 2), "%")
    print("Survival counts:")
    display(df["survived"].value_counts(dropna=False).rename_axis("survived").to_frame("count"))

if {"sex", "survived"}.issubset(df.columns):
    print("Survival rate by sex (%):")
    display((df.groupby("sex", observed=False)["survived"].mean() * 100).round(2))

if class_col and "survived" in df.columns:
    print("Survival rate by class (%):")
    display((df.groupby(class_col, observed=False)["survived"].mean() * 100).round(2))

### Conclusions to report
1. Survival was not evenly distributed across passenger groups.
2. Sex and passenger class showed clear associations with survival.
3. Fare was right-skewed and contained high-value outliers.
4. Age and cabin/deck variables had missing values that should be acknowledged.
5. Visual and statistical EDA identifies patterns, but it does not by itself prove causation.

**Limitations:** Results depend on the exact Titanic dataset version loaded. Cite the source used and report the actual row counts and missing-value counts produced by your notebook.